In [3]:
import json, time, random, threading, itertools, subprocess
from contextlib import contextmanager
from collections import defaultdict
from decimal import Decimal
import numpy as np
import pandas as pd
import psycopg2, psycopg2.errors, psycopg2.extras

import os
os.environ["PGURL"] = "postgresql://williamkim@localhost:5432/postgres"

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

# let psycopg2 accept numpy scalars pulled out of DataFrames
for _t in (np.int64, np.int32): psycopg2.extensions.register_adapter(_t, lambda v: psycopg2.extensions.AsIs(int(v)))
for _t in (np.float64, np.float32): psycopg2.extensions.register_adapter(_t, lambda v: psycopg2.extensions.AsIs(float(v)))

PGURL = os.environ.get("PGURL", "postgresql://postgres@localhost:5432/postgres")

def connect(autocommit=True):
    c = psycopg2.connect(PGURL, options="-c search_path=invest,public")
    c.autocommit = autocommit
    return c

conn = connect()

def q(sql, params=None, c=None):
    """Run SQL and return a DataFrame (NUMERIC columns become floats)."""
    with (c or conn).cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:
            return None
        df = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
    for col in df.columns:
        nn = df[col].dropna()
        if df[col].dtype == object and len(nn) and isinstance(nn.iloc[0], Decimal):
            df[col] = df[col].astype(float)
    return df

def run(sql, params=None, c=None):
    """Run a statement, return rows affected."""
    with (c or conn).cursor() as cur:
        cur.execute(sql, params)
        return cur.rowcount

def attempt(sql, params=None, c=None):
    """Execute and report errors (with SQLSTATE) instead of raising. Uses a savepoint inside a transaction."""
    c = c or conn
    in_txn = not c.autocommit
    cur = c.cursor()
    try:
        if in_txn: cur.execute("SAVEPOINT attempt_sp")
        cur.execute(sql, params)
        msg = f"OK (rows affected: {cur.rowcount})"
        if in_txn: cur.execute("RELEASE SAVEPOINT attempt_sp")
        return msg
    except psycopg2.Error as e:
        if in_txn: cur.execute("ROLLBACK TO SAVEPOINT attempt_sp")
        return f"{type(e).__name__} [{e.pgcode}]: {str(e).splitlines()[0]}"
    finally:
        cur.close()

@contextmanager
def sandbox():
    """A transaction that is always rolled back: experiments never touch the base data."""
    c = connect(autocommit=False)
    try:
        yield c
    finally:
        c.rollback()
        c.close()

def new_session():
    """Extra autocommit connection (use explicit BEGIN/COMMIT) for concurrency demos."""
    return connect(autocommit=True)

def explain(sql, c=None, analyze=False, buffers=False):
    opts = ["COSTS"] + (["ANALYZE"] if analyze else []) + (["BUFFERS"] if buffers else [])
    rows = q(f"EXPLAIN ({', '.join(opts)}) {sql}", c=c)
    for line in rows.iloc[:, 0]:
        print(line)

def ms(sql, c=None, n=5):
    """Best-of-n wall-clock time of a query in milliseconds."""
    best = 1e9
    for _ in range(n):
        s = time.perf_counter(); q(sql, c=c); best = min(best, time.perf_counter() - s)
    return best * 1000

SCHEMA = """
DROP SCHEMA IF EXISTS invest CASCADE;
CREATE SCHEMA invest;

CREATE TABLE invest.investor(
  investor_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  name        TEXT NOT NULL,
  email       TEXT NOT NULL UNIQUE,
  country     TEXT NOT NULL,
  joined      DATE NOT NULL,
  referred_by INT REFERENCES invest.investor(investor_id)
);
CREATE TABLE invest.instrument(
  symbol TEXT PRIMARY KEY,
  name   TEXT NOT NULL,
  sector TEXT NOT NULL,
  kind   TEXT NOT NULL CHECK (kind IN ('STOCK','ETF','BOND'))
);
CREATE TABLE invest.account(
  account_id   INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  investor_id  INT NOT NULL REFERENCES invest.investor(investor_id) ON DELETE CASCADE,
  kind         TEXT NOT NULL CHECK (kind IN ('INDIVIDUAL','JOINT','SUPER')),
  cash_balance NUMERIC(14,2) NOT NULL DEFAULT 0 CHECK (cash_balance >= 0),
  status       TEXT NOT NULL DEFAULT 'ACTIVE' CHECK (status IN ('ACTIVE','FROZEN'))
);
CREATE TABLE invest.trade(
  trade_id   INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  account_id INT NOT NULL REFERENCES invest.account(account_id),
  symbol     TEXT NOT NULL REFERENCES invest.instrument(symbol),
  side       TEXT NOT NULL CHECK (side IN ('BUY','SELL')),
  qty        INT NOT NULL CHECK (qty > 0),
  price      NUMERIC(12,2) NOT NULL CHECK (price > 0),
  traded_at  DATE NOT NULL
);
CREATE TABLE invest.price_history(
  symbol TEXT NOT NULL REFERENCES invest.instrument(symbol),
  day    DATE NOT NULL,
  close  NUMERIC(12,2) NOT NULL CHECK (close > 0),
  PRIMARY KEY (symbol, day)
);
CREATE TABLE invest.watchlist(
  investor_id INT NOT NULL REFERENCES invest.investor(investor_id),
  symbol      TEXT NOT NULL REFERENCES invest.instrument(symbol),
  added_on    DATE NOT NULL,
  PRIMARY KEY (investor_id, symbol)
);
"""
run(SCHEMA)

cur = conn.cursor()
cur.executemany("INSERT INTO investor VALUES (%s,%s,%s,%s,%s,%s)", [
    (1, "Ava Chen",   "ava@mail.com",  "Australia",   "2024-01-15", None),
    (2, "Ben Okafor", "ben@mail.com",  "Australia",   "2024-03-02", 1),
    (3, "Cara Singh", "cara@mail.com", "India",       "2024-03-20", 1),
    (4, "Dan Murphy", "dan@mail.com",  "New Zealand", "2024-06-11", 2),
    (5, "Eve Tanaka", "eve@mail.com",  "Japan",       "2024-06-11", 2),
    (6, "Finn Walsh", "finn@mail.com", "Australia",   "2025-01-09", 4),
    (7, "Gina Rossi", "gina@mail.com", "Italy",       "2025-02-14", None),
    (8, "Hugo Tran",  "hugo@mail.com", "Australia",   "2025-05-30", 7),
])
cur.executemany("INSERT INTO instrument VALUES (%s,%s,%s,%s)", [
    ("AAPL", "Apple Inc",           "tech",         "STOCK"),
    ("MSFT", "Microsoft Corp",      "tech",         "STOCK"),
    ("CBA",  "Commonwealth Bank",   "financials",   "STOCK"),
    ("BHP",  "BHP Group",           "materials",    "STOCK"),
    ("TLS",  "Telstra",             "telecom",      "STOCK"),
    ("VAS",  "Vanguard AU Shares",  "diversified",  "ETF"),
    ("VGB",  "Vanguard Govt Bonds", "fixed income", "BOND"),
])
cur.executemany("INSERT INTO account VALUES (%s,%s,%s,%s,%s)", [
    (1, 1, "INDIVIDUAL", 60000, "ACTIVE"), (2, 1, "SUPER", 80000, "ACTIVE"),
    (3, 2, "INDIVIDUAL", 55000, "ACTIVE"), (4, 2, "JOINT", 40000, "ACTIVE"),
    (5, 3, "INDIVIDUAL", 70000, "ACTIVE"), (6, 4, "INDIVIDUAL", 35000, "ACTIVE"),
    (7, 5, "SUPER", 90000, "ACTIVE"),      (8, 6, "INDIVIDUAL", 30000, "ACTIVE"),
    (9, 8, "INDIVIDUAL", 45000, "ACTIVE"),
])

random.seed(42)
DAYS = ["2026-10-19","2026-10-20","2026-10-21","2026-10-22","2026-10-23",
        "2026-10-26","2026-10-27","2026-10-28","2026-10-29","2026-10-30"]
BASE = {"AAPL":230.0,"MSFT":420.0,"CBA":165.0,"BHP":44.0,"TLS":4.8,"VAS":105.0,"VGB":52.0}
px = {}
for s, p in BASE.items():
    for d in DAYS:
        p = round(p * (1 + random.uniform(-0.02, 0.02)), 2)
        px[(s, d)] = p
cur.executemany("INSERT INTO price_history VALUES (%s,%s,%s)", [(s, d, c) for (s, d), c in px.items()])

held = defaultdict(int)
rows = []
for tid in range(1, 61):
    acct = random.randint(1, 9)
    sym = random.choice(list(BASE))
    day = random.choice(DAYS)
    side = "SELL" if (random.random() < 0.25 and held[(acct, sym)] > 0) else "BUY"
    qty = random.choice([10, 25, 50, 100, 200])
    if side == "SELL":
        qty = min(qty, held[(acct, sym)]); held[(acct, sym)] -= qty
    else:
        held[(acct, sym)] += qty
    rows.append((tid, acct, sym, side, qty, px[(sym, day)], day))
cur.executemany("INSERT INTO trade VALUES (%s,%s,%s,%s,%s,%s,%s)", rows)
cur.executemany("INSERT INTO watchlist VALUES (%s,%s,%s)", [
    (1, "AAPL", "2026-09-01"), (1, "CBA", "2026-09-03"), (2, "AAPL", "2026-09-10"),
    (3, "BHP", "2026-09-12"), (4, "VAS", "2026-09-15"), (5, "MSFT", "2026-09-20"),
    (5, "AAPL", "2026-09-21"), (7, "VGB", "2026-10-01"),
])
# identity columns: move each sequence past the explicit ids we loaded
for t, col in [("investor", "investor_id"), ("account", "account_id"), ("trade", "trade_id")]:
    cur.execute(f"SELECT setval(pg_get_serial_sequence('invest.{t}', '{col}'), (SELECT MAX({col}) FROM {t}))")
cur.close()

print(q("SELECT version()").iloc[0, 0].split(",")[0])
print("tables:", q("SELECT table_name FROM information_schema.tables WHERE table_schema='invest' ORDER BY 1").table_name.tolist())
print("trade rows:", q("SELECT COUNT(*) AS n FROM trade").n[0])

PostgreSQL 18.7 (Postgres.app) on aarch64-apple-darwin23.6.0
tables: ['account', 'instrument', 'investor', 'price_history', 'trade', 'watchlist']
trade rows: 60


## Setup: running investment-platform database

**Connection.** Set the `PGURL` environment variable (default `postgresql://postgres@localhost:5432/postgres`). The notebook creates and owns one schema called `invest`, so it never touches your other tables. Install the driver once with `pip install psycopg2-binary pandas`.

**Helpers**
- `q(sql, params)` runs a query and returns a DataFrame.
- `run(sql, params)` runs a statement and returns the number of rows affected.
- `attempt(sql, params)` runs a statement and reports a constraint error (with its SQLSTATE code) instead of raising.
- `sandbox()` opens a transaction that is **always rolled back** when the block ends. PostgreSQL supports transactional DDL, so even `CREATE TABLE`, `ALTER TABLE` and `CREATE ROLE` experiments disappear afterwards and the base data stays pristine.
- `explain(sql)` prints a query plan.
- `new_session()` opens an extra autocommit connection, used for multi-session concurrency demos in Chapter 6.

In [4]:
q("SELECT * FROM investor")

,investor_id,name,email,country,joined,referred_by
0,1,Ava Chen,ava@mail.com,Australia,2024-01-15,NaN
1,2,Ben Okafor,ben@mail.com,Australia,2024-03-02,1.0
2,3,Cara Singh,cara@mail.com,India,2024-03-20,1.0
3,4,Dan Murphy,dan@mail.com,New Zealand,2024-06-11,2.0
4,5,Eve Tanaka,eve@mail.com,Japan,2024-06-11,2.0
5,6,Finn Walsh,finn@mail.com,Australia,2025-01-09,4.0
6,7,Gina Rossi,gina@mail.com,Italy,2025-02-14,NaN
7,8,Hugo Tran,hugo@mail.com,Australia,2025-05-30,7.0
